# Train the free-text interpreter (GRPO + LoRA)
Runtime -> Change runtime type -> **T4 GPU**. Public/starter data only: do **not** upload private packs to Colab.

In [ ]:
!pip -q install 'trl>=0.14' peft transformers datasets accelerate
!pip -q uninstall -y torchao  # Colab's old torchao breaks newer peft; we don't use it
import torch; print(torch.cuda.get_device_name(0))

## Get the code
Either clone your public repo, or upload a zip made with:
`zip -r repo.zip . -x 'packs/*' 'runs/*' '.git/*' '__pycache__/*'`

In [ ]:
REPO_URL = ""  # e.g. https://github.com/you/your-repo.git ; leave empty to upload a zip
import os
if REPO_URL:
    !git clone $REPO_URL repo
else:
    from google.colab import files
    up = files.upload()
    !unzip -q -o {list(up)[0]} -d repo
%cd repo

In [ ]:
!python -m pytest -q
!python -m training.dataset --pack salt_exile --per-beat 40 --out runs/interp.jsonl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Train
Rough time on a free T4: **~1-3 hours for 300 steps** (each step generates 64 completions with plain HF generate, no vLLM). Do a **100-step test run first**. Checkpoints go to Google Drive every 25 steps and training resumes automatically if you re-run this cell after a disconnect.
Watch `reward` in the logs rise from ~0 toward ~0.8+. If it jumps to 1.0 immediately, the model is probably just emitting `{}` for everything: check samples.

In [ ]:
OUT = '/content/drive/MyDrive/desert-fates/interp-lora'
!python -m training.grpo_train --model Qwen/Qwen3-1.7B --data runs/interp.jsonl --out $OUT --steps 100 --save-steps 25  # raise --steps to 300 after a good test run

In [ ]:
!cd $OUT/.. && zip -qr /content/interp-lora.zip interp-lora
from google.colab import files; files.download('/content/interp-lora.zip')

## Evaluate on unseen phrasings
Run for your adapter and for `--adapter none` (base model) to compare.

In [ ]:
!python -m training.eval_interp --adapter $OUT
!python -m training.eval_interp --adapter none